In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "google-suite-t35-qa"
NOTEBOOK_PATH = "notebooks/qa/54-google-suite-t35-qa.ipynb"
TOWER = 35
TOWER_NAME = "Tower of Google Suite"
MIN_SCORE = 70
BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
results = []
def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))
def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""
print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 35: Tower of Google Suite


In [2]:
# Cell 1: Google app recipes exist
print("=== Google Suite App Recipes ===")

apps_dir = BROWSER_ROOT / "data" / "default" / "apps"

# P01: Gmail inbox triage recipe exists
gmail_recipe = read_file(apps_dir / "gmail-inbox-triage" / "recipe.json")
record("T35-P01", "Gmail inbox triage recipe exists with steps",
       '"id": "gmail-inbox-triage"' in gmail_recipe and '"steps"' in gmail_recipe,
       "gmail-inbox-triage/recipe.json")

# P02: Google Drive saver recipe exists with Drive scopes
drive_recipe = read_file(apps_dir / "google-drive-saver" / "recipe.json")
record("T35-P02", "Google Drive saver recipe exists with Drive scopes",
       '"drive.read.files"' in drive_recipe and '"drive.write.files"' in drive_recipe,
       "google-drive-saver has read + write Drive scopes")

# P03: Calendar brief recipe exists with calendar platform
cal_recipe = read_file(apps_dir / "calendar-brief" / "recipe.json")
record("T35-P03", "Calendar brief recipe exists with google-calendar platform",
       '"platform": "google-calendar"' in cal_recipe and '"calendar.read.events"' in cal_recipe,
       "calendar-brief platform=google-calendar, scope=calendar.read.events")

# P04: Google Search trends recipe exists
trends_recipe = read_file(apps_dir / "google-search-trends" / "recipe.json")
record("T35-P04", "Google Search trends recipe exists",
       '"id": "google-search-trends"' in trends_recipe and '"google.read.search"' in trends_recipe,
       "google-search-trends with google.read.search scope")

# P05: Morning brief recipe orchestrates cross-Google data
morning_recipe = read_file(apps_dir / "morning-brief" / "recipe.json")
record("T35-P05", "Morning brief recipe exists as multi-site orchestrator",
       '"type": "orchestrator"' in morning_recipe and '"platform": "multi-site"' in morning_recipe,
       "morning-brief orchestrates Gmail + Calendar + GitHub + Slack")

=== Google Suite App Recipes ===
PASS: Gmail inbox triage recipe exists with steps -- gmail-inbox-triage/recipe.json
PASS: Google Drive saver recipe exists with Drive scopes -- google-drive-saver has read + write Drive scopes
PASS: Calendar brief recipe exists with google-calendar platform -- calendar-brief platform=google-calendar, scope=calendar.read.events
PASS: Google Search trends recipe exists -- google-search-trends with google.read.search scope
PASS: Morning brief recipe exists as multi-site orchestrator -- morning-brief orchestrates Gmail + Calendar + GitHub + Slack


In [3]:
# Cell 2: Google Drive saver recipe structure
print("=== Google Drive Saver Recipe Details ===")

drive_recipe = read_file(BROWSER_ROOT / "data" / "default" / "apps" / "google-drive-saver" / "recipe.json")
drive_data = json.loads(drive_recipe) if drive_recipe else {}

# P06: Drive recipe navigates to Drive URL
steps_text = json.dumps(drive_data.get("steps", []))
record("T35-P06", "Drive recipe navigates to drive.google.com",
       "drive.google.com" in steps_text,
       "Step navigates to Google Drive web UI")

# P07: Drive recipe has navigate_to_folder action
record("T35-P07", "Drive recipe uses navigate_to_folder action",
       "navigate_to_folder" in steps_text,
       "Navigates to designated folder, creates if missing")

# P08: Drive recipe has upload_file action
record("T35-P08", "Drive recipe uses upload_file action",
       "upload_file" in steps_text,
       "Uploads local artifact to Google Drive")

# P09: Drive recipe saves evidence receipt to outbox
record("T35-P09", "Drive recipe saves upload receipt to outbox",
       "save_to_outbox" in steps_text and "drive-save-receipt" in steps_text,
       "Saves receipt with Drive file URL and hash")

# P10: Drive recipe has output_schema with uploaded + file_url
output_schema = drive_data.get("output_schema", {})
props = output_schema.get("properties", {})
record("T35-P10", "Drive recipe output_schema defines uploaded and file_url",
       "uploaded" in props and "file_url" in props,
       f"Output properties: {list(props.keys())}")

=== Google Drive Saver Recipe Details ===
PASS: Drive recipe navigates to drive.google.com -- Step navigates to Google Drive web UI
PASS: Drive recipe uses navigate_to_folder action -- Navigates to designated folder, creates if missing
PASS: Drive recipe uses upload_file action -- Uploads local artifact to Google Drive
PASS: Drive recipe saves upload receipt to outbox -- Saves receipt with Drive file URL and hash
PASS: Drive recipe output_schema defines uploaded and file_url -- Output properties: ['uploaded', 'file_title', 'file_url', 'target_folder', 'source_file_hash', 'timestamp']


In [4]:
# Cell 3: Calendar brief recipe structure
print("=== Calendar Brief Recipe Details ===")

cal_recipe = read_file(BROWSER_ROOT / "data" / "default" / "apps" / "calendar-brief" / "recipe.json")
cal_data = json.loads(cal_recipe) if cal_recipe else {}

# P11: Calendar recipe navigates to calendar.google.com
cal_steps = json.dumps(cal_data.get("steps", []))
record("T35-P11", "Calendar recipe navigates to calendar.google.com",
       "calendar.google.com" in cal_steps,
       "Navigates to Google Calendar day view")

# P12: Calendar recipe extracts events with title, time_range, location
record("T35-P12", "Calendar recipe extracts event title, time_range, and location",
       "title" in cal_steps and "time_range" in cal_steps and "location" in cal_steps,
       "Extracts structured event data from day view")

# P13: Calendar recipe detects conflicts between overlapping events
record("T35-P13", "Calendar recipe detects time conflicts between events",
       "detect_conflicts" in cal_steps,
       "Transform operation: detect_conflicts on start/end times")

# P14: Calendar recipe computes free slots
record("T35-P14", "Calendar recipe computes free time slots",
       "compute_free_slots" in cal_steps and "work_hours" in cal_steps,
       "Computes free slots within 09:00-18:00 work hours")

# P15: Calendar output_schema includes conflicts and free_slots arrays
cal_output = cal_data.get("output_schema", {}).get("properties", {})
record("T35-P15", "Calendar output_schema includes conflicts and free_slots",
       "conflicts" in cal_output and "free_slots" in cal_output,
       f"Output properties: {list(cal_output.keys())}")

=== Calendar Brief Recipe Details ===
PASS: Calendar recipe navigates to calendar.google.com -- Navigates to Google Calendar day view
PASS: Calendar recipe extracts event title, time_range, and location -- Extracts structured event data from day view
PASS: Calendar recipe detects time conflicts between events -- Transform operation: detect_conflicts on start/end times
PASS: Calendar recipe computes free time slots -- Computes free slots within 09:00-18:00 work hours
PASS: Calendar output_schema includes conflicts and free_slots -- Output properties: ['events', 'conflicts', 'free_slots', 'total_events', 'total_meetings_hours', 'timestamp']


In [5]:
# Cell 4: OAuth3 scopes and PrimeWiki for Google services
print("=== OAuth3 Scopes & PrimeWiki ===")

scopes_py = read_file(SRC / "oauth3" / "scopes.py")

# P16: Gmail scopes are registered (read.inbox, send.email, delete, label, draft, search)
gmail_scopes = ["gmail.read.inbox", "gmail.send.email", "gmail.delete.email",
                "gmail.label.apply", "gmail.draft.create", "gmail.search.messages"]
found = [s for s in gmail_scopes if f'"{s}"' in scopes_py]
record("T35-P16", "All 6 Gmail scopes registered in scopes.py",
       len(found) == 6,
       f"Found {len(found)}/6: {found}")

# P17: Gmail PrimeWiki directory exists with selectors, actions, urls
pw_gmail = BROWSER_ROOT / "data" / "default" / "primewiki" / "gmail"
pw_files = ["selectors.json", "actions.json", "urls.json"]
found_pw = [f for f in pw_files if (pw_gmail / f).exists()]
record("T35-P17", "Gmail PrimeWiki has selectors.json, actions.json, urls.json",
       len(found_pw) == 3,
       f"Found {len(found_pw)}/3: {found_pw}")

# P18: App store page references Google services
app_store_html = read_file(WEB / "app-store.html")
record("T35-P18", "App store page references Gmail in platform list",
       "Gmail" in app_store_html,
       "App store mentions Gmail in the platform description")

# P19: Budget files exist for all Google app recipes
google_apps = ["gmail-inbox-triage", "google-drive-saver", "calendar-brief", "google-search-trends"]
budgets_found = [a for a in google_apps if (BROWSER_ROOT / "data" / "default" / "apps" / a / "budget.json").exists()]
record("T35-P19", "Budget files exist for all 4 Google app recipes",
       len(budgets_found) == 4,
       f"Found {len(budgets_found)}/4: {budgets_found}")

# P20: Gmail evidence chain data exists (outbox/runs/evidence.json)
evidence_file = BROWSER_ROOT / "data" / "default" / "apps" / "gmail-inbox-triage" / "outbox" / "runs" / "evidence.json"
evidence_content = read_file(evidence_file)
record("T35-P20", "Gmail triage has evidence.json in outbox with hashes",
       "evidence_hash" in evidence_content and "esign_hash" in evidence_content,
       "SHA-256 evidence hash + e-sign hash in evidence.json")

=== OAuth3 Scopes & PrimeWiki ===
PASS: All 6 Gmail scopes registered in scopes.py -- Found 6/6: ['gmail.read.inbox', 'gmail.send.email', 'gmail.delete.email', 'gmail.label.apply', 'gmail.draft.create', 'gmail.search.messages']
PASS: Gmail PrimeWiki has selectors.json, actions.json, urls.json -- Found 3/3: ['selectors.json', 'actions.json', 'urls.json']
PASS: App store page references Gmail in platform list -- App store mentions Gmail in the platform description
PASS: Budget files exist for all 4 Google app recipes -- Found 4/4: ['gmail-inbox-triage', 'google-drive-saver', 'calendar-brief', 'google-search-trends']
PASS: Gmail triage has evidence.json in outbox with hashes -- SHA-256 evidence hash + e-sign hash in evidence.json


In [6]:
# Cell 5: Evidence summary
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 35: Tower of Google Suite
Probes: 20 | Passed: 20 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: a5d8a543bb6f9f3e

{
  "surface": "google-suite-t35-qa",
  "tower": 35,
  "tower_name": "Tower of Google Suite",
  "timestamp": "2026-03-06T11:29:34.704286",
  "total_probes": 20,
  "passed": 20,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "a5d8a543bb6f9f3e"
}
